# Unified Video Analysis Pipeline (Colab)

Runs the Streamlit app from **https://github.com/SudeepIITM/MicroVLM_Edge1** via a Cloudflare tunnel.

**Before running:** set the runtime to GPU -> *Runtime > Change runtime type > Hardware accelerator > GPU (T4)*.

Run the cells top-to-bottom. The last cell prints a public `https://*.trycloudflare.com` URL.

Pipeline components:
1. **Summary** - Qwen3-VL + Q-Former LoRA checkpoint
2. **Multi-class** - EnhancedTemporalAdapterModel using the generated summary
3. **Binary** - SimpleAdapter + LoRA ensemble

## 1. Clone the repository

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/SudeepIITM/MicroVLM_Edge1.git"
REPO_DIR = "/content/MicroVLM_Edge1"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)

print(f"Repository ready at {REPO_DIR}")
subprocess.run(["ls", "-la", REPO_DIR])

## 2. Install dependencies

In [ ]:
import sys

req = os.path.join(REPO_DIR, "requirements.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req], check=True)
print("Dependencies installed.")

## 3. Mount Drive and configure model paths

Edit the four paths below to match your Google Drive layout.

- `MULTICLASS_MODEL_DIR` must contain `multiclass_temporal_adapter_qwen3_instruct_improved.pt` and `multiclass_label_encoders.pkl`.
- `SUMMARIZATION_CHECKPOINT` is the Q-Former LoRA `checkpoint-400` directory.
- `BINARY_MODEL_DIR` must contain `input_dim.pkl`, `simple_adapter.pt`, `lora_model.pt`.
- `VIDEO_DIR` is a folder of sample videos (optional).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ---- MODEL PATHS (from training pipeline models_path.txt) ----
QWEN_MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
# Q-Former LoRA summarization checkpoint
SUMMARIZATION_CHECKPOINT = "/content/drive/MyDrive/Project_VLM/ucf_qwen_v9_qformer/checkpoint-400"
# Multi-class dir: contains multiclass_temporal_adapter_qwen3_instruct_improved.pt + multiclass_label_encoders.pkl
MULTICLASS_MODEL_DIR = "/content/drive/MyDrive/models_ucf_v2/hpo_best_models"
# Binary weights source dir (training-pipeline naming: instruct_*)
BINARY_DRIVE_DIR = "/content/drive/MyDrive/models_ucf_v2"
# Sample videos
VIDEO_DIR = "/content/drive/MyDrive/Project_VLM/All_Videos"
# --------------------------------------------------------------

# Copy + RENAME binary weights into the repo's model/ dir.
# binary_ensemble.py expects: input_dim.pkl, simple_adapter.pt, lora_model.pt
# Training pipeline saves:     instruct_input_dim.pkl, instruct_simple_adapter.pt, lora_model.pt
import shutil
BINARY_MODEL_DIR = os.path.join(REPO_DIR, "model")
os.makedirs(BINARY_MODEL_DIR, exist_ok=True)

binary_file_map = {
    "instruct_input_dim.pkl": "input_dim.pkl",
    "instruct_simple_adapter.pt": "simple_adapter.pt",
    "lora_model.pt": "lora_model.pt",
}
for src_name, dst_name in binary_file_map.items():
    src = os.path.join(BINARY_DRIVE_DIR, src_name)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(BINARY_MODEL_DIR, dst_name))
        print(f"  Copied {src_name} -> {dst_name}")
    else:
        # Fall back to already-correct names if the instruct_* variant is absent
        alt = os.path.join(BINARY_DRIVE_DIR, dst_name)
        if os.path.exists(alt):
            shutil.copy(alt, os.path.join(BINARY_MODEL_DIR, dst_name))
            print(f"  Copied {dst_name}")
        else:
            print(f"  MISSING {src_name} / {dst_name} (check BINARY_DRIVE_DIR)")

# Export env vars consumed by pipeline.py / binary_ensemble.py
os.environ["QWEN_MODEL_ID"] = QWEN_MODEL_ID
os.environ["SUMMARIZATION_CHECKPOINT"] = SUMMARIZATION_CHECKPOINT
os.environ["MULTICLASS_MODEL_DIR"] = MULTICLASS_MODEL_DIR
os.environ["BINARY_MODEL_DIR"] = BINARY_MODEL_DIR
os.environ["VIDEO_DIR"] = VIDEO_DIR
os.environ["PYTHONHASHSEED"] = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

print("\nEnvironment configured:")
for k in ["QWEN_MODEL_ID", "SUMMARIZATION_CHECKPOINT", "MULTICLASS_MODEL_DIR", "BINARY_MODEL_DIR", "VIDEO_DIR"]:
    print(f"  {k} = {os.environ[k]}")

# Quick existence check for the multi-class + summary files
mc_ckpt = os.path.join(MULTICLASS_MODEL_DIR, "multiclass_temporal_adapter_qwen3_instruct_improved.pt")
mc_enc = os.path.join(MULTICLASS_MODEL_DIR, "multiclass_label_encoders.pkl")
print("\nFile checks:")
print(f"  multiclass ckpt : {'OK' if os.path.exists(mc_ckpt) else 'MISSING'}  {mc_ckpt}")
print(f"  multiclass enc  : {'OK' if os.path.exists(mc_enc) else 'MISSING'}  {mc_enc}")
print(f"  summary ckpt    : {'OK' if os.path.isdir(SUMMARIZATION_CHECKPOINT) else 'MISSING'}  {SUMMARIZATION_CHECKPOINT}")

## 4. Download the Cloudflare tunnel binary

In [ ]:
cloudflared_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
subprocess.run(["wget", "-q", cloudflared_url, "-O", "cloudflared"], check=True)
subprocess.run(["chmod", "+x", "cloudflared"], check=True)
subprocess.run(["./cloudflared", "--version"])

## 5. Launch Streamlit + public tunnel

Give it ~60s to boot and load the Qwen3-VL model. The public URL appears at the bottom.

In [ ]:
import time

PORT = 8501
app_path = os.path.join(REPO_DIR, "streamlit_app_unified.py")

streamlit_proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", app_path,
        "--server.port", str(PORT),
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ],
    stdout=open("/content/streamlit.log", "w"),
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

print("Waiting for Streamlit to start...")
for i in range(12):
    time.sleep(5)
    if streamlit_proc.poll() is not None:
        print("ERROR: Streamlit terminated. Recent log:")
        print(open("/content/streamlit.log").read()[-1500:])
        break
    print(f"  Still starting... ({(i+1)*5}s elapsed)")

print("\nRecent Streamlit log:")
print("".join(open("/content/streamlit.log").readlines()[-30:]))

In [ ]:
print("=== Public URL will appear below (look for *.trycloudflare.com) ===\n")

cloudflared_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in cloudflared_proc.stdout:
    print(line.strip())
    if "https://" in line and "trycloudflare.com" in line:
        print(f"\n*** PUBLIC URL: {line.strip()} ***\n")
        print("Access your app at the URL above!")
        break